# Parallel Deterministic Selection with Dynamic Data

In [ ]:
import math
import random

# simulated distributed nodes
N0 = [3, 8, 15]
N1 = [1, 9, 20]
N2 = [4, 7, 11]

nodes = [N0, N1, N2]

print("Distributed Nodes:")
for idx, node in enumerate(nodes):
    print(f"Node {idx}: {node}")


## Part A: Deterministic Parallel Selection

In [ ]:
def find_middle(values):
    ordered = values.copy()
    ordered.sort()

    return ordered[len(ordered) // 2]


def deterministic_pivot(values):
    if len(values) <= 5:
        return find_middle(values)

    chunks = []
    start = 0

    while start < len(values):
        chunks.append(values[start:start + 5])
        start += 5

    local_medians = []

    for chunk in chunks:
        local_medians.append(find_middle(chunk))

    return deterministic_pivot(local_medians)


def flatten_nodes(nodes):
    merged = []

    for node in nodes:
        merged.extend(node)

    return merged


def kth_query(nodes, k):
    values = flatten_nodes(nodes)

    if k < 1 or k > len(values):
        return None

    return selection_recursive(values, k - 1)


def selection_recursive(values, index):
    if len(values) <= 5:
        ordered = sorted(values)
        return ordered[index]

    pivot = deterministic_pivot(values)

    left_side = []
    pivot_bucket = []
    right_side = []

    for value in values:
        if value < pivot:
            left_side.append(value)
        elif value > pivot:
            right_side.append(value)
        else:
            pivot_bucket.append(value)

    if index < len(left_side):
        return selection_recursive(left_side, index)

    if index < len(left_side) + len(pivot_bucket):
        return pivot

    return selection_recursive(
        right_side,
        index - len(left_side) - len(pivot_bucket)
    )


## Dynamic Updates

In [ ]:
def add_value(nodes, value):
    target_node = min(nodes, key=len)
    target_node.append(value)


def remove_value(nodes, target):
    for node in nodes:
        if target in node:
            node.remove(target)
            return True

    return False


## Example Queries

In [ ]:
print("\nInitial Query:")
answer = kth_query(nodes, 5)
print(f"5th smallest element -> {answer}")

print("\nInsert Operation:")
add_value(nodes, 6)

for idx, node in enumerate(nodes):
    print(f"Node {idx}: {node}")

answer = kth_query(nodes, 5)
print(f"Updated 5th smallest element -> {answer}")

print("\nDelete Operation:")
remove_value(nodes, 8)

for idx, node in enumerate(nodes):
    print(f"Node {idx}: {node}")


## Part B: Local Summaries

In [ ]:
def local_summary(node):
    if len(node) == 0:
        return {
            "size": 0,
            "median": None
        }

    return {
        "size": len(node),
        "median": find_middle(node)
    }


print("\nLocal Summaries:")

node_summaries = []

for node in nodes:
    node_summaries.append(local_summary(node))

for idx, info in enumerate(node_summaries):
    print(f"Node {idx}: {info}")


## Part C: Complexity

In [ ]:
n = len(flatten_nodes(nodes))
p = len(nodes)

print("\nComplexity Analysis")

print("Selection Query:")
print(f"Work Complexity -> O(n)")
print(f"Parallel Time -> O(n/p + log p)")
print(f"Communication Cost -> O(p)")

print("\nInsert/Delete:")
print("Average Update Cost -> O(1)")
print("Minimal Communication Between Nodes")


## Part D: Correctness Test

In [ ]:
print("\nCorrectness Verification")

merged = flatten_nodes(nodes)
expected = sorted(merged)

for k in range(1, len(expected) + 1):
    result = kth_query(nodes, k)

    if result == expected[k - 1]:
        print(f"k = {k} -> Correct")
    else:
        print(f"k = {k} -> Wrong")


## Part E: Trade-offs

In [ ]:
print("\nTrade-offs Discussion")

print("1. Exact selection gives accurate results but increases computation.")
print("2. Approximate pivots reduce communication overhead.")
print("3. Incremental updates are cheaper than full recomputation.")
print("4. Frequent adversarial updates may force pivot rebuilding.")


## Bonus Ideas

In [ ]:
print("\nBonus Features")

print("- Fault tolerance can replicate summaries across nodes")
print("- Approximate quantiles improve scalability")
print("- Amortized updates avoid rebuilding after every operation")


## Lower Bound Discussion

In [ ]:
print("\nLower Bound Discussion")

print("Dynamic distributed selection has theoretical lower bounds")
print("because processors must exchange information to maintain global ordering.")

print("As updates increase,")
print("communication overhead becomes unavoidable in exact selection systems.")

print("Approximate approaches reduce communication")
print("but may sacrifice perfect accuracy.")
